<div align="right" style="text-align: right"><i>Peter Norvig<br>May 2026</i></div>

# Fractions approximating π

The number π is irrational, which means that it cannot be represented exactly by a fraction. But some fractions, such as 22/7, are known to come close to π. In this notebook, we look for fractions that approximate π even more closely than 22/7.

If we want to know *what's the best fraction approximation to π with a denominator no more than a given number of digits?* we can try every possible denominator up to the given number of digits, compute the best numerator (and hence the best fraction) for each denominator, and choose the resulting fraction that minimizes the distance to the target, π (or if we want, any target):

In [1]:
from fractions import Fraction
from math import pi

def approximation(target=pi, digits=6) -> Fraction:
    """The fraction best approximating `target` whose denominator has no more than `digits` digits."""
    denominators = range(1, 10 ** digits)
    fractions    = (Fraction(round(target * d),   d) for d in denominators)
    return min(fractions, key=lambda x: abs(target - x))

For example, here are the best approximations with one- and three-digit denominators:

In [2]:
approximation(pi, 1)

Fraction(22, 7)

In [3]:
approximation(pi, 3)

Fraction(355, 113)

We can make a pretty report showing the best approximations to π with 1, 2, 3, 4, and 5-digit denominators:

In [4]:
def report(approximator=approximation, target=pi, d_range=range(1, 6)) -> str:
    """For each `d` in `d_range`, print a description of how close 
    the fraction `approximator(target, d)` is to `target`."""
    for d in d_range:
        r = approximator(target, d)
        print(f'{r:^22} = {r:.25f} (error {r-target:+6.0e})')

In [5]:
report()

         22/7          = 3.1428571428571428571428571 (error +1e-03)
        311/99         = 3.1414141414141414141414141 (error -2e-04)
       355/113         = 3.1415929203539823008849558 (error +3e-07)
       355/113         = 3.1415929203539823008849558 (error +3e-07)
     312689/99532      = 3.1415926536189366233975003 (error +3e-11)


(The approximation 355/113 is so good that no 4-digit-denominator is better. You can think of 3550/1130 as the best 4-digit approximation.)

# Faster, Better Approximations

A 5-digit denominator is the most that `approximation` can handle in under a second of run time.  

We can get better approximations much faster by iteratively improving a guess:
- The zeroth approximation to π is formed by rounding to the nearest integer, **3**.
- The first approximation to π is formed by adding 3 to an approximation of the remainder after rounding, 0.14159...
  - Rounding the remainder would give us 0, so that doesn't help.
  - Instead of rounding the remainder, use (1 / approximation(1 / remainder)).
  - 1 / 0.14159... rounds to 7 (with remainder   0.06251...) so the first approximation is **3 + (1/7) = 22/7**.
- The second approximation to π is formed by adding in an approximation of 0.06251...
  - 1 / 0.06251... rounds to 16, so the second approximation is **3 + (1/(7 + 1/16)) = 355/113**.
  - You can repeat the approximation process to any depth.
  
We can implement it like this:

In [6]:
PI = Fraction("3.14159265358979323846264338327950288419716939937510582097494459230781640628620899862803")

def fast_approximation(target: Fraction, depth: int) -> Fraction:
    """Approximate `target` by taking the whole part plus an approximation to the remainder.
    Repeat `depth` times (or stop when there is no remainder).
    Use (1 / fast_approximation(1 / remainder)), because remainder < 1 and (1 / remainder) > 1."""
    whole = round(target)
    remainder = target - whole
    if depth == 0 or remainder == 0:
        return Fraction(whole)
    else:
        return whole + (1 / fast_approximation(1 / remainder, depth - 1))

This gives an approximation accurate to 22 digits after 14 iterations, and takes just milliseconds to run:

In [7]:
report(fast_approximation, PI, range(14))

          3            = 3.0000000000000000000000000 (error -1e-01)
         22/7          = 3.1428571428571428571428571 (error +1e-03)
       355/113         = 3.1415929203539823008849558 (error +3e-07)
     104348/33215      = 3.1415926539214210447087159 (error +3e-10)
     312689/99532      = 3.1415926536189366233975003 (error +3e-11)
    1146408/364913     = 3.1415926535914039784825424 (error +2e-12)
   5419351/1725033     = 3.1415926535898153832419438 (error +2e-14)
  80143857/25510582    = 3.1415926535897926593756269 (error -6e-16)
  245850922/78256779   = 3.1415926535897931602832772 (error -8e-17)
 411557987/131002976   = 3.1415926535897932578264482 (error +2e-17)
 1068966896/340262731  = 3.1415926535897932353925649 (error -3e-18)
 2549491779/811528438  = 3.1415926535897932390140098 (error +6e-19)
6167950454/1963319607  = 3.1415926535897932383863775 (error -8e-20)
21053343141/6701487259 = 3.1415926535897932384623817 (error -3e-22)


It turns out there is a name for this approach: it is the [continued fraction approximation to π](https://en.wikipedia.org/wiki/Pi#Continued_fractions). 

Note: it is more traditional to use `whole = int(target)` rather than `whole = round(target)` to generate the continued fractions. Using `round` gives better fractions in fewer itera, but using `int` has the nice property that all the bits we add are positive.